# <center>  **<span style="font-size:80px;">Visualización</span>** </center>

In [ ]:
import pandas as pd
import geopandas as gpd
import sys
import os

from pathlib import Path

sys.path.append(os.path.abspath(os.path.join("..")))
from src.config import DatasetKeys, Paths
Paths.init_project()

In [ ]:
mapas_path = Path("./AMAEM/mapas/")
mapas_path.mkdir(parents=True, exist_ok=True)

# Visualización

In [ ]:
def generar_mapa_interactivo(
    ruta_csv,
    ruta_geojson,
    variable_mapa,
    columnas_tooltip,
    agregaciones,
    nombre_archivo_salida,
    cmap="YlOrRd",
    convertir_variable_a_entero=False
):
    """
    Genera un mapa interactivo (Folium) cruzando datos tabulares con geometrías reales.
    """
    # 1. CARGA Y AGREGACIÓN DE DATOS
    df_raw = pd.read_csv(ruta_csv)

    # Filtramos solo las columnas que realmente existan en el CSV
    dict_agg = {k: v for k, v in agregaciones.items() if k in df_raw.columns}

    # Agrupamos por barrio para tener una única fila por polígono
    df_datos = df_raw.groupby(DatasetKeys.BARRIO).agg(dict_agg).reset_index()

    # Limpiamos el nombre del barrio (ej: "10-FLORIDA BAJA" -> "FLORIDA BAJA")
    df_datos['barrio_limpio'] = df_datos[DatasetKeys.BARRIO].str.split('-', n=1).str[-1].str.strip()

    # 2. CARGAMOS GEOMETRÍAS REALES
    # Usamos directamente geopandas para leer el archivo JSON de Entidades
    gdf_areas = gpd.read_file(ruta_geojson)

    # 3. CRUCE DE DATOS Y GEOMETRÍA
    gdf_final = gdf_areas.merge(
        df_datos, 
        left_on='DENOMINACI', 
        right_on='barrio_limpio', 
        how='inner'
    )

    # ConvertiMOS de UTM (25830) a GPS (4326) para que Folium lo posicione bien
    gdf_final = gdf_final.to_crs(epsg=4326)

    # Calculamos el percentil 95 para tratar los supuestos 'Outliers'
    #vmax = gdf_final[variable_mapa].quantile(0.95)
    vmax = gdf_final[variable_mapa].max()

    # Transformación opcional: Convertir la variable a entero si el usuario lo pide
    if convertir_variable_a_entero:
        gdf_final[variable_mapa] = gdf_final[variable_mapa].astype(int)

    # 4. VISUALIZACIÓN (Folium / Leaflet)
    mapa_hackathon = gdf_final.explore(
        column=variable_mapa,
        cmap=cmap,                # Paleta configurable
        vmin=0,                   
        vmax=vmax,                
        tooltip=columnas_tooltip, # Tooltips configurables
        popup=True,               
        tiles="CartoDB dark_matter", 
        style_kwds={              
            "fillOpacity": 0.6,   
            "weight": 0.8,        
            "color": "#444444"    
        },
        highlight_kwds={          
            "fillOpacity": 0.9, 
            "weight": 2, 
            "color": "#D6D8D8"
        },
        name="Infraestructura y Consumo"
    )

    # Guardamos el resultado
    ruta_salida = mapas_path / nombre_archivo_salida
    mapa_hackathon.save(ruta_salida)

    print(f"Mapa '{nombre_archivo_salida}' generado con éxito en: {ruta_salida}")
    
    return mapa_hackathon

## Ejemplo 1: Mapa del Consumo Total

In [ ]:
agregaciones_base = {
    DatasetKeys.CONSUMO: 'sum',
    DatasetKeys.NUM_CONTRATOS: 'sum',
    DatasetKeys.CONSUMO_RATIO: 'mean',
    DatasetKeys.PCT_VT_BARRIO: 'mean',
    DatasetKeys.NUM_VT_BARRIO: 'sum'
}

mapa_consumo = generar_mapa_interactivo(
    ruta_csv=Paths.PROC_CSV_DIR / "not_scaled.csv",
    ruta_geojson=Paths.RAW_JSON_ENTIDADES_POBLACION,
    variable_mapa=DatasetKeys.CONSUMO,
    columnas_tooltip=["DENOMINACI", DatasetKeys.CONSUMO, DatasetKeys.NUM_CONTRATOS],
    agregaciones=agregaciones_base,
    nombre_archivo_salida="mapa_consumo_total.html",
    cmap="YlGnBu", 
    convertir_variable_a_entero=True # El consumo tiene sentido como entero
)

## Ejemplo 2: Mapa de Presión Turística (Viviendas Turísticas)

In [ ]:
mapa_turismo = generar_mapa_interactivo(
    ruta_csv=Paths.PROC_CSV_DIR / "not_scaled.csv",
    ruta_geojson=Paths.RAW_JSON_ENTIDADES_POBLACION,
    variable_mapa=DatasetKeys.PCT_VT_BARRIO,
    columnas_tooltip=["DENOMINACI", DatasetKeys.PCT_VT_BARRIO, DatasetKeys.NUM_VT_BARRIO],
    agregaciones=agregaciones_base,
    nombre_archivo_salida="mapa_presion_turistica.html",
    cmap="plasma", # Resalta mucho las zonas turísticas
    convertir_variable_a_entero=False # Mantenemos los decimales para el porcentaje
)